# Notebook 4: Datasets & DataLoaders
**Estimated time:** 30-45 min
**Prerequisites:** Notebooks 1-3

---
## What you will learn
1. Why we use DataLoader instead of manually batching
2. How to create a custom `Dataset` class
3. How `DataLoader` handles batching, shuffling, and parallel loading
4. `torchvision` transforms for image preprocessing
5. How to load real data from CSV files

## 1. Why DataLoaders?

In Notebook 3, we fed the entire dataset through the model at once (full-batch gradient descent).
This only works for tiny datasets that fit in RAM/VRAM.

Real datasets (ImageNet: 1.2M images, Common Crawl: petabytes of text) must be:
- **Loaded in chunks** (batches) — feed 32 or 64 images at a time
- **Shuffled** — so the model doesn't learn the order
- **Preprocessed** — resize, normalize, augment
- **Loaded in parallel** — use multiple CPU workers while GPU trains

`DataLoader` handles all of this automatically.

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt

## 2. Custom Dataset Class

Every dataset you create must:
1. Inherit from `torch.utils.data.Dataset`
2. Implement `__len__(self)` — returns the number of samples
3. Implement `__getitem__(self, idx)` — returns one sample by index

That's it. PyTorch's `DataLoader` does the rest.

In [ ]:
class TabularDataset(Dataset):
    '''A dataset that wraps numpy arrays of features and labels.'''

    def __init__(self, X, y):
        # Store as float32 tensors (PyTorch default float type)
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


# Create dummy data
np.random.seed(42)
X_np = np.random.randn(500, 10).astype(np.float32)
y_np = np.random.randint(0, 3, size=500)   # 3 classes

dataset = TabularDataset(X_np, y_np)

print(f'Dataset size      : {len(dataset)}')
print(f'One sample shapes : X={dataset[0][0].shape}, y={dataset[0][1].shape}')
print(f'First sample      : X={dataset[0][0][:3]}..., y={dataset[0][1]}')

## 3. DataLoader

Wrap any `Dataset` in a `DataLoader` to get automatic batching, shuffling, and multi-process loading.

In [ ]:
loader = DataLoader(
    dataset,
    batch_size=32,
    shuffle=True,       # re-shuffle every epoch
    num_workers=0,      # 0 = main process; >0 = parallel workers (use 2-4 on GPU machines)
    drop_last=False     # if True, drop final incomplete batch
)

print(f'Number of batches: {len(loader)}')  # ceil(500 / 32) = 16

# Iterate one batch
batch_X, batch_y = next(iter(loader))
print(f'Batch X shape: {batch_X.shape}')
print(f'Batch y shape: {batch_y.shape}')

In [ ]:
# Typical training loop WITH DataLoader
import torch.nn as nn
import torch.optim as optim

model = nn.Sequential(nn.Linear(10, 32), nn.ReLU(), nn.Linear(32, 3))
optimizer = optim.Adam(model.parameters())
criterion = nn.CrossEntropyLoss()

EPOCHS = 5
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for batch_X, batch_y in loader:    # <-- DataLoader yields batches
        optimizer.zero_grad()
        logits = model(batch_X)
        loss   = criterion(logits, batch_y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    avg_loss = total_loss / len(loader)
    print(f'Epoch {epoch+1}: avg loss = {avg_loss:.4f}')

## 4. Train / Val / Test Split

Use `torch.utils.data.random_split` to split a dataset into train, val, and test subsets.

In [ ]:
from torch.utils.data import random_split

# 70% train, 15% val, 15% test
total = len(dataset)
n_train = int(0.7 * total)
n_val   = int(0.15 * total)
n_test  = total - n_train - n_val

train_set, val_set, test_set = random_split(
    dataset, [n_train, n_val, n_test],
    generator=torch.Generator().manual_seed(42)
)

train_loader = DataLoader(train_set, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_set,   batch_size=64, shuffle=False)
test_loader  = DataLoader(test_set,  batch_size=64, shuffle=False)

print(f'Train: {len(train_set)} | Val: {len(val_set)} | Test: {len(test_set)}')

## 5. torchvision Transforms (Image Preprocessing)

For image datasets, `torchvision.transforms` provides a composable pipeline.

In [ ]:
import torchvision.transforms as T
from torchvision.datasets import MNIST

# Compose multiple transforms into a pipeline
transform = T.Compose([
    T.ToTensor(),              # PIL Image/numpy → tensor, scales [0,255] → [0,1]
    T.Normalize((0.1307,), (0.3081,))  # subtract mean, divide by std
])

# Download and load MNIST
train_mnist = MNIST(root='./data', train=True,  download=True, transform=transform)
test_mnist  = MNIST(root='./data', train=False, download=True, transform=transform)

print(f'Train size: {len(train_mnist)}')
print(f'Test  size: {len(test_mnist)}')
print(f'Image shape: {train_mnist[0][0].shape}')  # (1, 28, 28)
print(f'Classes: {train_mnist.classes}')

# Create DataLoaders
train_loader_mnist = DataLoader(train_mnist, batch_size=64, shuffle=True,  num_workers=0)
test_loader_mnist  = DataLoader(test_mnist,  batch_size=64, shuffle=False, num_workers=0)

# Visualize some images
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for i, ax in enumerate(axes.flat):
    img, label = train_mnist[i]
    ax.imshow(img.squeeze(), cmap='gray')
    ax.set_title(str(label))
    ax.axis('off')
plt.suptitle('MNIST samples')
plt.tight_layout()
plt.show()

## 6. Data Augmentation Transforms

Augmentation artificially increases dataset size by applying random transformations during training.
This reduces overfitting — the model sees slightly different versions of the same image each epoch.

In [ ]:
# Training transforms (with augmentation)
train_transform = T.Compose([
    T.RandomHorizontalFlip(p=0.5),
    T.RandomRotation(degrees=15),
    T.ColorJitter(brightness=0.2, contrast=0.2),
    T.ToTensor(),
    T.Normalize((0.5,), (0.5,))
])

# Test/val transforms (NO augmentation — just normalize)
test_transform = T.Compose([
    T.ToTensor(),
    T.Normalize((0.5,), (0.5,))
])

print('Train transforms: random flip + rotate + color jitter + normalize')
print('Test  transforms: normalize only (deterministic!)')
print()
print('IMPORTANT: never apply random augmentation to val/test sets.')
print('You need a fair, consistent benchmark to measure model quality.')

## 7. Custom Dataset from CSV

A common real-world pattern: loading data from CSV files.

In [ ]:
import csv, os

# First, create a sample CSV file to work with
csv_path = './data/sample.csv'
os.makedirs('./data', exist_ok=True)

np.random.seed(0)
n = 200
X_csv = np.random.randn(n, 3)
y_csv = (X_csv[:, 0] + X_csv[:, 1] > 0).astype(int)

with open(csv_path, 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['feat1', 'feat2', 'feat3', 'label'])
    for i in range(n):
        writer.writerow([*X_csv[i], y_csv[i]])

print(f'Written {n} rows to {csv_path}')

In [ ]:
class CSVDataset(Dataset):
    def __init__(self, path):
        data = []
        with open(path) as f:
            reader = csv.DictReader(f)
            for row in reader:
                feats  = [float(row['feat1']), float(row['feat2']), float(row['feat3'])]
                label  = int(row['label'])
                data.append((feats, label))
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        feats, label = self.data[idx]
        return torch.tensor(feats, dtype=torch.float32), torch.tensor(label)


csv_dataset = CSVDataset(csv_path)
csv_loader  = DataLoader(csv_dataset, batch_size=16, shuffle=True)

X_batch, y_batch = next(iter(csv_loader))
print(f'CSV batch X: {X_batch.shape}')
print(f'CSV batch y: {y_batch.shape}')

---
## YOUR TURN — Mini Project

**Task:** Build a complete data pipeline and train a model on MNIST.

**Steps:**
1. Create `DataLoader`s for MNIST train and test (batch_size=128, num_workers=0)
2. Build a model: `784 → 256 → 128 → 10` (28×28 images flattened)
   - Add `nn.Flatten()` as the first layer — it turns `(B, 1, 28, 28)` → `(B, 784)`
3. Train for 5 epochs
4. After each epoch, compute and print train loss AND test accuracy
5. Plot the per-epoch test accuracy

**Bonus:** Explore `torchvision.transforms.RandomAffine` — does it improve test accuracy?

In [ ]:
# YOUR CODE HERE

# 1. DataLoaders (MNIST already downloaded above)
# train_loader = ...
# test_loader  = ...

# 2. Model (start with nn.Flatten())

# 3. Loss and optimizer

# 4. Training loop

# 5. Plot test accuracy per epoch